# Trivial baselines on track 1

The floor every real system has to clear: a majority-label predictor and a seeded random predictor, scored on the dev set with the official scorer.

**What this notebook does**

1. Loads the English train and dev files.
2. Predicts every dev instance with the training majority label, and again with a uniform random label from a fixed seed.
3. Scores both prediction files with the organizers' scorer, run exactly as CodaBench runs it.
4. Records the run: seed, data commit, repository commit and the exact command.

**Where the results go**

| Output | Path |
|---|---|
| Run record | `experiments/2026-09-29-trivial-baselines-track1/config.json` |
| Predictions | `experiments/2026-09-29-trivial-baselines-track1/predictions/` |
| Scorer reports | `experiments/2026-09-29-trivial-baselines-track1/scores/<system>/` |
| Results table | `experiments/2026-09-29-trivial-baselines-track1/results.md` |

**How to run**

```bash
PYTHONIOENCODING=utf-8 uv run jupyter execute notebooks/assignment-1/01_trivial_baselines.ipynb
```

Needs the task repository cloned at `datasets/dico-nli/`; the command is in `datasets/dico-nli.json`.

## 1. Setup

Imports and the run parameters. `RUN_DATE` names the folder the run is recorded under; a rerun replicates into the same folder, and `OVERWRITE = True` makes that explicit.

In [ ]:
from datetime import date

import pandas as pd

from dico_nli.baselines.trivial import majority_label, predict_majority, predict_random
from dico_nli.data import DATA_ROOT, load_reference
from dico_nli.experiments import experiment_dir, write_config, write_results_table
from dico_nli.scoring import score, write_predictions

TRACK = 1
SEED = 13
RUN_DATE = date(2026, 9, 29)   # the folder this run is recorded under; reruns replicate into it
OVERWRITE = True               # a rerun is deliberate here; the guard exists for scripts that might loop

## 2. Data

Train supplies the majority label. Dev is the course test set: it is scored and never used for any choice.

In [ ]:
train = load_reference(TRACK, "train")
dev = load_reference(TRACK, "dev")
gold = DATA_ROOT / "dev" / f"dico_nli_dev_track{TRACK}_reference.csv"
out = experiment_dir(f"trivial-baselines-track{TRACK}", day=RUN_DATE, overwrite=OVERWRITE)
print(f"train {len(train)}  dev {len(dev)}  majority label in train: {majority_label(train)}")

## 3. Predict and score

Two prediction files in the official `instance_id,label` format, each scored by the official scorer.

Track-1 train has 881 FORWARD_ENTAILMENT and 881 BACKWARD_ENTAILMENT instances. `majority_label` resolves the tie by canonical label order, so the majority system predicts FORWARD_ENTAILMENT everywhere, and its SoftCons is 0 by construction: the reverse of forward is backward.

In [ ]:
systems = {
    "majority": predict_majority(train, dev),
    f"random (seed {SEED})": predict_random(dev, SEED),
}
rows = {}
for name, predictions in systems.items():
    key = name.split(" ")[0]
    prediction_path = write_predictions(out / "predictions" / f"{key}.csv", predictions)
    rows[name] = score(gold, prediction_path, out / "scores" / key)

pd.DataFrame({name: {"weighted_f1": s.weighted_f1, "soft_cons": s.soft_cons, "hard_cons": s.hard_cons}
              for name, s in rows.items()}).T.round(4)

## 4. Run record

`config.json` records the seed, the data commit, this repository's commit (and whether the tree was dirty) and the exact command. `results.md` holds the table above in the form the report cites.

In [ ]:
write_config(
    out, seed=SEED, track=TRACK,
    command="PYTHONIOENCODING=utf-8 uv run jupyter execute notebooks/assignment-1/01_trivial_baselines.ipynb",
    extra={"majority_label": majority_label(train), "train_size": len(train), "dev_size": len(dev), "systems": list(systems)},
)
print(write_results_table(out, rows, title=f"Trivial baselines, track {TRACK} dev").read_text(encoding="utf-8"))